# K-Means Samsung: estabilidad de submuestreo y decisiones metodológicas

Notebook/script independiente para comparar 50, 100, 200 y 300 repeticiones,
variando la fracción de submuestreo: 50%, 60%, 70%, 80%, 90%, 95% y 99%.

Reproduce la metodología del notebook de K-Means:
- W TRAIN previamente winsorizada desde R, una fila por response_id.
- Escalado Z-score con desviación estándar muestral (ddof=1), ajustado en TRAIN completo.
- Referencia: KMeans(n_init=30, max_iter=400, seed=123) sobre todo TRAIN.
- Submuestreo sin reemplazo, semilla determinista, reajuste K-Means en cada repetición.
- ARI entre etiquetas de referencia y de submuestra, sobre exactamente los mismos clientes.
- Jaccard por grupo emparejado de manera óptima mediante algoritmo húngaro.

Importante: NO es posible recorrer todas las fracciones matemáticamente posibles;
se usa una malla amplia con un extremo de estrés (50%) y uno casi total (99%).
Repeticiones 50/100/200/300 son PREFIJOS de la misma secuencia de 300 ejecuciones:
el experimento de 50 está contenido en el de 100, etc. No se recalcula cuatro veces.

También se examina sensibilidad a la semilla y a n_init, que es distinta de la
estabilidad ante cambios en los clientes. Las salidas NO modifican el notebook original.

## 1. Configuración y carga de datos

Ajusta RUTA_CSV_MANUAL si el CSV no está en data/kmeans o junto al archivo.
MODO_CANDIDATOS: 'candidatos' (3 modelos elegidos en exploración),
'candidatos_y_k4' (los candidatos + K=4 en A/B/C) o 'todos_k' (27 modelos; costoso).
La versión predeterminada compara 5 modelos, aproximadamente 10.500 ajustes
adicionales (5 modelos x 7 fracciones x 300 repeticiones); puede tardar.
PRUEBA_RAPIDA=True ejecuta pocos ajustes para verificar el ambiente y guarda
los resultados en una subcarpeta diferente, nunca confundirlos con resultados finales.

In [1]:
from __future__ import annotations

import hashlib
import json
import math
import warnings
from datetime import datetime, timezone
from pathlib import Path
from config_matrices import MIN_MONETARY_T1, TAG, CARPETA_SALIDA, ruta_matrices, cargar_W_train

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sklearn
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

SEMILLA = 123
N_INIT = 30
MAX_ITER = 400
TOL = 1e-4
ALGORITMO = "lloyd"

N_REPETICIONES = [50, 100, 200, 300]
FRACCIONES = [0.50, 0.60, 0.70, 0.80, 0.90, 0.95, 0.99]

MODO_CANDIDATOS = "candidatos"

CANDIDATOS_MANUALES = [
    ("A_RFM_3", 2),
    ("B_RFM_ratio_4", 3),
    ("B_RFM_ratio_4", 4),
    ("C_W_6", 2),
    ("C_W_6", 3),
    ("C_W_6", 5),
]

COMPROBAR_INICIALIZACION = True
N_INIT_ALTERNATIVOS = [10, 30, 50]
SEMILLAS_INICIALIZACION = [123, 124, 125, 126, 127, 321, 456, 789, 2024, 2026]
GUARDAR_CADA = 25
PRUEBA_RAPIDA = False
RUTA_CSV_MANUAL = f"../../data/matrices/matrices_data{TAG}.csv"
RUTA_EXPLORACION_MANUAL = None  # opcional: exploracion_A_B_C_k2_k10.csv

RFM = [
    "recency_days_2018_Samsung",
    "frequency_2018_Samsung",
    "monetary_2018_Samsung",
]
SEIS_VARIABLES = RFM + [
    "dollar_avg_12m",
    "dollar_top1_12m",
    "dollar_avg12_top1_ratio",
]
CONFIGURACIONES = {
    "A_RFM_3": RFM,
    "B_RFM_ratio_4": RFM + ["dollar_avg12_top1_ratio"],
    "C_W_6": SEIS_VARIABLES,
}

## 2. Funciones de reproducibilidad, selección y métricas

Los mismos valores escalados de TRAIN completo se mantienen al evaluar submuestras.
Esto reproduce el procedimiento original. Reestandarizar independientemente cada
submuestra sería otro experimento y mezclaría fuentes de variabilidad.

In [2]:
def ruta_csv() -> Path:
    return ruta_matrices(RUTA_CSV_MANUAL)  # data/matrices/matrices_data{TAG}


def buscar_exploracion(p_csv: Path) -> Path | None:
    if RUTA_EXPLORACION_MANUAL is not None:
        p = Path(RUTA_EXPLORACION_MANUAL).expanduser().resolve()
        if not p.is_file():
            raise FileNotFoundError(p)
        return p
    nombre = "exploracion_A_B_C_k2_k10.csv"
    bases = [CARPETA_SALIDA]
    posibilidades = []
    for base in bases:
        posibilidades += [base / f"resultados_kmeans_python{TAG}" / nombre, base / nombre]
    for p in posibilidades:
        if p.is_file():
            return p.resolve()
    return None


def candidatos_desde_exploracion(p: Path) -> list[tuple[str, int]]:
    tabla = pd.read_csv(p)
    columnas = {
        "configuracion", "k", "n_clusters_reales", "alerta_cluster_menor_3pct", "silhouette_euclidean"
    }
    if not columnas.issubset(tabla.columns):
        raise ValueError(f"El archivo de exploración no tiene columnas: {columnas-set(tabla.columns)}")
    seleccion = []
    for conf in CONFIGURACIONES:
        grupo = tabla.loc[tabla.configuracion == conf]
        if grupo.empty:
            raise ValueError(f"No hay datos de exploración para {conf}")
        alertas = grupo["alerta_cluster_menor_3pct"].astype(str).str.lower().isin(["true", "1"])
        reales = grupo["n_clusters_reales"] == grupo["k"]
        elegibles = grupo.loc[reales & ~alertas]
        if elegibles.empty:
            warnings.warn(f"{conf}: no hay soluciones sin alerta de clústeres pequeños")
            elegibles = grupo.loc[reales]
        if elegibles.empty:
            raise ValueError(f"No hay soluciones válidas para {conf}")
        fila = elegibles.sort_values("silhouette_euclidean", ascending=False).iloc[0]
        seleccion.append((conf, int(fila["k"])))
    referencia = ("B_RFM_ratio_4", 4)
    if referencia not in seleccion:
        seleccion.append(referencia)
    return seleccion


def seleccionar_candidatos(p_explo: Path | None) -> list[tuple[str, int]]:
    if CANDIDATOS_MANUALES is not None:
        seleccion = [(str(c), int(k)) for c, k in CANDIDATOS_MANUALES]
    elif p_explo is not None:
        seleccion = candidatos_desde_exploracion(p_explo)
    else:
        warnings.warn(
            "No se encontró tabla de exploración. Se usarán K=4 para A/B/C; "
            "para reproducir la lista corta original, indica RUTA_EXPLORACION_MANUAL."
        )
        seleccion = [(c, 4) for c in CONFIGURACIONES]
    if MODO_CANDIDATOS == "candidatos_y_k4":
        seleccion += [(c, 4) for c in CONFIGURACIONES]
    elif MODO_CANDIDATOS == "todos_k":
        seleccion += [(c, k) for c in CONFIGURACIONES for k in range(2, 11)]
    elif MODO_CANDIDATOS != "candidatos":
        raise ValueError("MODO_CANDIDATOS debe ser candidatos, candidatos_y_k4 o todos_k")
    seleccion = list(dict.fromkeys(seleccion))
    for conf, k in seleccion:
        if conf not in CONFIGURACIONES or not 2 <= k <= 10:
            raise ValueError(f"Candidato inválido: {conf}, K={k}")
    return seleccion


def cargar_train(p_csv: Path) -> pd.DataFrame:
    df = cargar_W_train(p_csv)
    if "response_id" not in df:
        raise ValueError("Falta response_id")
    ids = df["response_id"].str.strip()
    if ids.isna().any() or ids.eq("").any() or ids.duplicated().any():
        raise ValueError("response_id contiene valores ausentes, vacíos o repetidos")
    if not set(SEIS_VARIABLES).issubset(df.columns):
        raise ValueError(f"Faltan variables: {set(SEIS_VARIABLES)-set(df.columns)}")
    for v in SEIS_VARIABLES:
        df[v] = pd.to_numeric(df[v], errors="raise")
    if not np.isfinite(df[SEIS_VARIABLES].to_numpy(dtype=float)).all():
        raise ValueError("Las seis variables deben ser numéricas, finitas y completas")
    if (df[SEIS_VARIABLES].std(ddof=1) <= 0).any():
        raise ValueError("Alguna variable tiene varianza nula")
    return df


def escalar_train(df: pd.DataFrame, variables: list[str]) -> np.ndarray:
    x = df[variables].to_numpy(dtype=float)
    return (x - x.mean(axis=0)) / x.std(axis=0, ddof=1)


def crear_kmeans(k: int, semilla: int, n_init: int = N_INIT) -> KMeans:
    return KMeans(
        n_clusters=k, n_init=n_init, max_iter=MAX_ITER,
        random_state=semilla, tol=TOL, algorithm=ALGORITMO,
    )


def jaccard_emparejado(etiquetas_ref: np.ndarray, etiquetas_nuevas: np.ndarray, k: int) -> np.ndarray:
    """Misma definición que notebook original: coincidencia óptima mediante Hungarian."""
    matriz = np.zeros((k, k), dtype=float)
    for i in range(k):
        ref = etiquetas_ref == i
        for j in range(k):
            nuevo = etiquetas_nuevas == j
            union = np.count_nonzero(ref | nuevo)
            matriz[i, j] = np.count_nonzero(ref & nuevo) / union if union else 0.0
    filas, columnas = linear_sum_assignment(-matriz)
    salida = np.zeros(k, dtype=float)
    salida[filas] = matriz[filas, columnas]
    return salida


def huella_archivo(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()


def firma_experimento(p_csv: Path, candidatos, fracciones, repeticiones) -> dict:
    return {
        "version_metodo": "1.0-submuestreo-hungarian-prefijos",
        "sha256_csv": huella_archivo(p_csv),
        "candidatos": [list(v) for v in candidatos],
        "fracciones": fracciones,
        "repeticiones": repeticiones,
        "semilla": SEMILLA,
        "n_init": N_INIT,
        "max_iter": MAX_ITER,
        "tol": TOL,
        "algoritmo": ALGORITMO,
        "comprobar_inicializacion": COMPROBAR_INICIALIZACION,
        "n_init_alternativos": N_INIT_ALTERNATIVOS if COMPROBAR_INICIALIZACION else [],
        "semillas_alternativas": SEMILLAS_INICIALIZACION if COMPROBAR_INICIALIZACION else [],
        "version_sklearn": sklearn.__version__,
        "version_numpy": np.__version__,
        "version_scipy": scipy.__version__,
    }


def asegurar_manifest(carpeta: Path, firma: dict) -> None:
    carpeta.mkdir(parents=True, exist_ok=True)
    ruta = carpeta / "manifest_experimento.json"
    if ruta.exists():
        anterior = json.loads(ruta.read_text(encoding="utf-8"))
        if anterior["firma"] != firma:
            raise RuntimeError(
                f"La configuración cambió respecto a la ejecución anterior en {carpeta}. "
                "Renombra/mueve esa carpeta o cambia SALIDAS antes de repetir. "
                "No se mezclarán resultados incompatibles."
            )
    else:
        ruta.write_text(json.dumps({
            "fecha_utc": datetime.now(timezone.utc).isoformat(),
            "firma": firma,
            "nota": "Una corrida de 300 repeticiones se reutiliza para 50, 100 y 200. "
                    "Fracciones alternativas son escenarios independientes.",
        }, ensure_ascii=False, indent=2), encoding="utf-8")

## 3. Evaluación incremental ARI/Jaccard en cada fracción

Cada combinación configuración-K-fracción conserva sus repeticiones en un CSV
propio. Se guardan checkpoints periódicos y se reanuda una ejecución interrumpida
sin repetir los ajustes ya completos. A 80% conserva los números iniciales de
la función de submuestreo original; las otras fracciones emplean semillas separadas.

In [3]:
def detalle_estabilidad(
    X: np.ndarray, etiquetas_ref: np.ndarray, conf: str, k: int,
    indice_candidato: int, fraccion: float, max_repeticiones: int, carpeta: Path
) -> pd.DataFrame:
    frac_pct = int(round(fraccion * 1000))
    nombre = f"detalle_{conf}_K{k}_f{frac_pct:03d}.csv"
    ruta = carpeta / nombre
    n = len(X)
    n_sub = max(k + 1, int(math.floor(fraccion * n)))
    if n_sub >= n:
        raise ValueError("El submuestreo debe retirar al menos un cliente")

    semilla_base = SEMILLA + 1000 * (indice_candidato + 1)
    semilla_muestreo = semilla_base if abs(fraccion - 0.80) < 1e-12 else semilla_base + 100_000 + frac_pct
    rng = np.random.default_rng(semilla_muestreo)
    cols_jac = [f"jaccard_cluster_{i+1}" for i in range(k)]

    previos = pd.read_csv(ruta) if ruta.exists() else pd.DataFrame()
    n_prev = len(previos)
    if n_prev:
        esperadas = list(range(1, n_prev + 1))
        if not np.array_equal(previos["repeticion"].to_numpy(dtype=int), esperadas):
            raise ValueError(f"Repeticiones corruptas o fuera de orden en {ruta}")
        if n_prev > max_repeticiones:
            raise ValueError("El detalle tiene más repeticiones que el experimento")
        if not all(c in previos for c in ["ARI", *cols_jac]):
            raise ValueError(f"El detalle está incompleto: {ruta}")
    if n_prev:
        print(f"    {conf} K={k} fracción={fraccion:.0%}: reanudando desde repetición {n_prev+1}", flush=True)

    pendiente = []
    for rep in range(1, max_repeticiones + 1):
        # El generador avanza de la misma forma aun cuando la corrida se reanuda.
        posiciones = rng.choice(n, size=n_sub, replace=False)
        if rep <= n_prev:
            continue
        nuevo = crear_kmeans(k, SEMILLA + rep).fit_predict(X[posiciones])
        ref_sub = etiquetas_ref[posiciones]
        ari = float(adjusted_rand_score(ref_sub, nuevo))
        jac = jaccard_emparejado(ref_sub, nuevo, k)
        fila = {
            "configuracion": conf,
            "k": k,
            "fraccion": fraccion,
            "n_submuestra": n_sub,
            "repeticion": rep,
            "semilla_muestreo": semilla_muestreo,
            "semilla_kmeans": SEMILLA + rep,
            "k_real_submuestra": len(np.unique(nuevo)),
            "ARI": ari,
        }
        fila.update({cols_jac[i]: float(jac[i]) for i in range(k)})
        pendiente.append(fila)
        if len(pendiente) >= GUARDAR_CADA or rep == max_repeticiones:
            pd.DataFrame(pendiente).to_csv(
                ruta, mode="a", header=not ruta.exists(), index=False
            )
            pendiente = []
            print(f"    {conf} K={k} fracción={fraccion:.0%}: {rep}/{max_repeticiones}", flush=True)
    resultado = pd.read_csv(ruta)
    if len(resultado) != max_repeticiones:
        raise AssertionError(f"Se esperaban {max_repeticiones} repeticiones en {ruta}")
    return resultado


def resumen_para_prefijos(detalles: list[pd.DataFrame], repeticiones: list[int]):
    filas = []
    filas_cluster = []
    for det in detalles:
        conf, k, frac = str(det.configuracion.iat[0]), int(det.k.iat[0]), float(det.fraccion.iat[0])
        cols_j = [f"jaccard_cluster_{i+1}" for i in range(k)]
        for R in repeticiones:
            sub = det.iloc[:R]
            ari = sub.ARI.to_numpy(dtype=float)
            promedios = sub[cols_j].mean(axis=0)
            sds = sub[cols_j].std(axis=0, ddof=1)
            sd_ari = float(np.std(ari, ddof=1)) if R > 1 else np.nan
            sem_ari = sd_ari / np.sqrt(R) if R > 1 else np.nan
            filas.append({
                "configuracion": conf,
                "k": k,
                "n_variables": len(CONFIGURACIONES[conf]),
                "fraccion": frac,
                "porcentaje_submuestra": round(frac * 100, 2),
                "n_submuestra": int(sub.n_submuestra.iat[0]),
                "n_repeticiones": R,
                "ARI_media": float(np.mean(ari)),
                "ARI_sd": sd_ari,
                "ARI_mediana": float(np.median(ari)),
                "ARI_p05": float(np.quantile(ari, 0.05)),
                "ARI_p95": float(np.quantile(ari, 0.95)),
                "ARI_error_estandar_MC": sem_ari,
                "ARI_MC_IC95_inferior_aprox": float(np.mean(ari) - 1.96 * sem_ari),
                "ARI_MC_IC95_superior_aprox": float(np.mean(ari) + 1.96 * sem_ari),
                "Jaccard_media_clusters": float(promedios.mean()),
                "Jaccard_min_cluster": float(promedios.min()),
                "Jaccard_sd_entre_clusters": float(promedios.std(ddof=1)) if k > 1 else 0.0,
                "Jaccard_min_observado": float(sub[cols_j].min().min()),
                "n_ajustes_con_menos_k": int((sub.k_real_submuestra < k).sum()),
            })
            for i in range(k):
                filas_cluster.append({
                    "configuracion": conf, "k": k,
                    "fraccion": frac, "n_repeticiones": R,
                    "cluster": i + 1,
                    "Jaccard_media": float(promedios.iloc[i]),
                    "Jaccard_sd": float(sds.iloc[i]),
                    "Jaccard_p05": float(sub[cols_j[i]].quantile(.05)),
                })
    resumen = pd.DataFrame(filas).sort_values(["configuracion", "k", "fraccion", "n_repeticiones"])
    por_cluster = pd.DataFrame(filas_cluster)
    referencia_300 = resumen[resumen.n_repeticiones == max(repeticiones)][
        ["configuracion", "k", "fraccion", "ARI_media", "Jaccard_min_cluster"]
    ].rename(columns={
        "ARI_media": "ARI_media_Rmax",
        "Jaccard_min_cluster": "Jaccard_min_cluster_Rmax"
    })
    convergencia = resumen.merge(referencia_300, on=["configuracion", "k", "fraccion"], validate="many_to_one")
    convergencia["delta_abs_ARI_vs_Rmax"] = (convergencia.ARI_media - convergencia.ARI_media_Rmax).abs()
    convergencia["delta_abs_Jaccard_min_vs_Rmax"] = (
        convergencia.Jaccard_min_cluster - convergencia.Jaccard_min_cluster_Rmax
    ).abs()
    return resumen, por_cluster, convergencia

## 4. Sensibilidad adicional a `n_init` y a las semillas aleatorias

Cambiar los clientes y cambiar la inicialización no es lo mismo. Este diagnóstico
repite el ajuste en todo TRAIN con 10 semillas y 3 valores de `n_init`.
Compara ARI frente a la solución de referencia y el cambio relativo de WSS.
No utiliza TEST ni modifica la solución de referencia.

In [4]:
def inicializaciones(df: pd.DataFrame, candidatos, datos_escalados, referencias) -> pd.DataFrame:
    registros = []
    for conf, k in candidatos:
        X = datos_escalados[conf]
        modelo_ref = referencias[(conf, k)]
        for n_init in N_INIT_ALTERNATIVOS:
            for s in SEMILLAS_INICIALIZACION:
                modelo = crear_kmeans(k, s, n_init=n_init).fit(X)
                registros.append({
                    "configuracion": conf, "k": k,
                    "semilla": s, "n_init": n_init,
                    "ARI_vs_referencia": float(adjusted_rand_score(modelo_ref.labels_, modelo.labels_)),
                    "inercia": float(modelo.inertia_),
                    "diferencia_pct_inercia_vs_ref": 100 * float((modelo.inertia_ / modelo_ref.inertia_) - 1),
                    "n_iteraciones": int(modelo.n_iter_),
                })
    return pd.DataFrame(registros)


def graficar_estabilidad(resumen: pd.DataFrame, salida: Path):
    rmax = int(resumen.n_repeticiones.max())
    for (conf, k), t in resumen.groupby(["configuracion", "k"], sort=False):
        fig, ejes = plt.subplots(1, 2, figsize=(13, 4.7))
        for R, grupo in t.groupby("n_repeticiones"):
            grupo = grupo.sort_values("fraccion")
            x = grupo.porcentaje_submuestra.to_numpy()
            ejes[0].plot(x, grupo.ARI_media, marker="o", label=f"{R} rep.")
            ejes[1].plot(x, grupo.Jaccard_min_cluster, marker="o", label=f"{R} rep.")
        ejes[0].set_title("ARI promedio vs. fracción")
        ejes[1].set_title("Jaccard del grupo menos estable")
        for eje in ejes:
            eje.set_xlabel("Clientes incluidos en la submuestra (%)")
            eje.set_ylabel("Índice (más alto = más estable)")
            eje.set_ylim(-0.05, 1.05)
            eje.grid(alpha=.25)
            eje.legend(fontsize=8)
        fig.suptitle(f"Robustez al porcentaje y al número de repeticiones — {conf}, K={k}")
        fig.tight_layout()
        fig.savefig(salida / f"estabilidad_fracciones_{conf}_K{k}.png", dpi=170, bbox_inches="tight")
        plt.close(fig)

    # Tabla de estabilidad MC por configuración; no implica validación poblacional.
    resumen_max = resumen.loc[resumen.n_repeticiones == rmax]
    for (conf, k), t in resumen_max.groupby(["configuracion", "k"], sort=False):
        fig, ax = plt.subplots(figsize=(9, 4.6))
        t = t.sort_values("fraccion")
        x = t.porcentaje_submuestra.to_numpy()
        y = t.ARI_media.to_numpy()
        err = t.ARI_error_estandar_MC.to_numpy() * 1.96
        ax.errorbar(x, y, yerr=err, fmt="o-", capsize=4)
        ax.set(xlabel="Fracción de TRAIN (%)", ylabel="ARI medio", title=f"Incertidumbre Monte Carlo — {conf}, K={k}, R={rmax}")
        ax.set_ylim(-0.05, 1.05)
        ax.grid(alpha=.25)
        fig.tight_layout()
        fig.savefig(salida / f"error_montecarlo_{conf}_K{k}.png", dpi=170, bbox_inches="tight")
        plt.close(fig)

## 5. Ejecutar y guardar resultados

Si se interrumpe, vuelve a ejecutar: el script detecta los CSV parciales y continúa.
La tabla `convergencia_vs_300.csv` muestra cuánto cambian las medias al pasar de
50/100/200 a 300 repeticiones. El intervalo de Monte Carlo NO es un intervalo
de confianza para la población de clientes ni un contraste de hipótesis.

In [5]:
def main():
    p_csv = ruta_csv()
    df = cargar_train(p_csv)
    p_explo = buscar_exploracion(p_csv)
    candidatos = seleccionar_candidatos(p_explo)

    if PRUEBA_RAPIDA:
        repeticiones = [5, 10]
        fracciones = [0.60, 0.80, 0.95]
        candidatos = candidatos[: min(3, len(candidatos))]
        carpeta_nombre = "estabilidad_PRUEBA_RAPIDA_no_tesis"
    else:
        repeticiones = N_REPETICIONES
        fracciones = FRACCIONES
        carpeta_nombre = "estabilidad_multifraccion_50_100_200_300"

    if sorted(set(repeticiones)) != repeticiones or min(repeticiones) < 2:
        raise ValueError("N_REPETICIONES debe ser ascendente, sin duplicados y >=2")
    if len(set(fracciones)) != len(fracciones) or not all(0 < f < 1 for f in fracciones):
        raise ValueError("FRACCIONES debe contener valores únicos entre 0 y 1")

    salida = CARPETA_SALIDA / f"resultados_kmeans_python{TAG}" / carpeta_nombre
    firma = firma_experimento(p_csv, candidatos, fracciones, repeticiones)
    asegurar_manifest(salida, firma)

    datos = {conf: escalar_train(df, CONFIGURACIONES[conf]) for conf in CONFIGURACIONES}
    referencias = {}
    for conf, k in candidatos:
        referencias[(conf, k)] = crear_kmeans(k, SEMILLA).fit(datos[conf])

    print("\nANÁLISIS DE ESTABILIDAD DE K-MEANS — SAMSUNG\n", flush=True)
    print(f"CSV: {p_csv}")
    print(f"TRAIN: {len(df)} clientes")
    print(f"Tabla de exploración: {p_explo or 'NO ENCONTRADA'}")
    print(f"Candidatos: {candidatos}")
    print(f"Fracciones: {fracciones}")
    print(f"Repeticiones acumulativas: {repeticiones}")
    print(f"Total ajustes de submuestra (si inicia desde cero): {len(candidatos)*len(fracciones)*max(repeticiones):,}")
    print(f"Salidas: {salida}\n", flush=True)

    detalles = []
    for indice, (conf, k) in enumerate(candidatos):
        X = datos[conf]
        ref = referencias[(conf, k)].labels_
        print(f"Candidato {indice+1}/{len(candidatos)}: {conf}, K={k}", flush=True)
        for f in fracciones:
            detalles.append(detalle_estabilidad(
                X, ref, conf, k, indice, f, max(repeticiones), salida
            ))

    detalle_total = pd.concat(detalles, ignore_index=True)
    detalle_total.to_csv(salida / "estabilidad_detalle_todos_escenarios.csv", index=False)
    resumen, por_cluster, convergencia = resumen_para_prefijos(detalles, repeticiones)
    resumen.to_csv(salida / "estabilidad_resumen_por_fraccion_y_repeticiones.csv", index=False)
    por_cluster.to_csv(salida / "estabilidad_jaccard_por_cluster_fraccion.csv", index=False)
    convergencia.to_csv(salida / f"convergencia_vs_{max(repeticiones)}.csv", index=False)
    graficar_estabilidad(resumen, salida)

    if COMPROBAR_INICIALIZACION:
        comparacion_seeds = inicializaciones(df, candidatos, datos, referencias)
        comparacion_seeds.to_csv(salida / "sensibilidad_semilla_y_n_init.csv", index=False)
        tabla_seeds = (
            comparacion_seeds.groupby(["configuracion", "k", "n_init"], as_index=False)
            .agg(ARI_min=("ARI_vs_referencia", "min"),
                 ARI_medio=("ARI_vs_referencia", "mean"),
                 variacion_max_pct_inercia=("diferencia_pct_inercia_vs_ref", "max"))
        )
        tabla_seeds.to_csv(salida / "resumen_semilla_y_n_init.csv", index=False)

    tabla_80 = resumen.loc[
        (np.isclose(resumen.fraccion, 0.80)) &
        (resumen.n_repeticiones == max(repeticiones))
    ]
    print(f"\nRESUMEN 80% / {max(repeticiones)} repeticiones (si están incluidos):", flush=True)
    if not tabla_80.empty:
        print(tabla_80[
            ["configuracion", "k", "ARI_media", "ARI_sd", "Jaccard_min_cluster", "ARI_error_estandar_MC"]
        ].round(4).to_string(index=False), flush=True)

    cambios = convergencia.loc[convergencia.n_repeticiones < max(repeticiones)]
    print(f"\nMáxima variación de las medias respecto a R={max(repeticiones)} (por escenario):")
    if not cambios.empty:
        print("  ARI:", round(float(cambios.delta_abs_ARI_vs_Rmax.max()), 5))
        print("  Jaccard mínimo:", round(float(cambios.delta_abs_Jaccard_min_vs_Rmax.max()), 5))
    print("\nArchivos escritos en:", salida.resolve())
    return resumen, por_cluster, convergencia

In [6]:
if __name__ == "__main__":
    resumen_estabilidad_multi, jaccard_estabilidad_multi, convergencia_estabilidad_multi = main()

C:\Users\jeam2\AppData\Local\Temp\ipykernel_28076\3337838911.py:75: UserWarning: La matriz no trae 'response_id': se crea un identificador por posición (train_00001, ...). Sirve para K-Means, pero NO permite cruzar con otras tablas; el orden de filas debe ser el de train_ids.
  df = cargar_W_train(p_csv)



ANÁLISIS DE ESTABILIDAD DE K-MEANS — SAMSUNG

CSV: C:\Users\jeam2\Downloads\tesis\cod_grupo2\cod_grupo2\data\Matrices\matrices_data_min80.xlsx
TRAIN: 1752 clientes
Tabla de exploración: C:\Users\jeam2\Downloads\tesis\cod_grupo2\cod_grupo2\cluster_tag\cluster_tag\resultados_kmeans_python_min80\exploracion_A_B_C_k2_k10.csv
Candidatos: [('A_RFM_3', 2), ('B_RFM_ratio_4', 3), ('B_RFM_ratio_4', 4), ('C_W_6', 2), ('C_W_6', 3), ('C_W_6', 5)]
Fracciones: [0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99]
Repeticiones acumulativas: [50, 100, 200, 300]
Total ajustes de submuestra (si inicia desde cero): 12,600
Salidas: c:\Users\jeam2\Downloads\tesis\cod_grupo2\cod_grupo2\cluster_tag\cluster_tag\resultados_kmeans_python_min80\estabilidad_multifraccion_50_100_200_300

Candidato 1/6: A_RFM_3, K=2
    A_RFM_3 K=2 fracción=50%: 25/300
    A_RFM_3 K=2 fracción=50%: 50/300
    A_RFM_3 K=2 fracción=50%: 75/300
    A_RFM_3 K=2 fracción=50%: 100/300
    A_RFM_3 K=2 fracción=50%: 125/300
    A_RFM_3 K=2 fracción=50%: 1

## 6. Cómo interpretar y defender el análisis

1. **Repeticiones (50/100/200/300):** observa convergencia de las medias de
   ARI y del Jaccard del clúster menos estable. Que las medias cambien poco al
   aumentar R indica que el error de simulación Monte Carlo es pequeño; NO
   garantiza que la estructura de los datos sea estable.
2. **Fracciones (50%, 60%, 70%, 80%, 90%, 95%, 99%):** si ARI y Jaccard se
   mantienen altos incluso cuando se retira 30%-50% de TRAIN, la partición
   resiste perturbaciones relativamente grandes. Una fracción de 99% es un
   control de perturbación mínima y suele inflar la estabilidad; NO debe
   usarse sola para decidir.
3. **Comparación de K y variables:** cada candidato tiene su propia partición
   de referencia; ARI=1 significa estabilidad frente a ESA partición, no
   superioridad comercial ni mayor Silhouette. Comparar también calidad,
   tamaño y perfiles, y recordar que Jaccard del clúster menos estable importa.
4. **Semillas y n_init:** si los ARI frente a referencia son bajos, K-Means es
   sensible a las inicializaciones, aun si las submuestras resultan estables.